In [1]:
import numpy as np
import glob
import os




def calculate_overall_rrms(measured_list, modeled_list):
    """
    Calculates the Mean RRMS across multiple bias curves 
    as defined by Eq. 5.1 and 5.2 in the paper.
    """
    rrms_k_list = []
    
    # Loop over each individual bias curve (k)
    for meas, mod in zip(measured_list, modeled_list):
        # 1. Handle shape mismatch by truncating to the shorter length (Nk)
        Nk = min(len(meas), len(mod))
        meas = meas[:Nk]
        mod = mod[:Nk]
        
        # 2. Equation 5.1 - Numerator: RMS of the absolute error
        numerator = np.sqrt(np.mean((mod - meas) ** 2))
        
        # 3. Equation 5.1 - Denominator: Mean of the absolute measured data
        # Adding a tiny offset (1e-12) to prevent division by zero if a curve is entirely empty
        denominator = np.mean(np.abs(meas))
        if denominator == 0:
            denominator = 1e-9
            
        # Calculate RRMS for the individual kth curve
        rrms_k = numerator / denominator
        rrms_k_list.append(rrms_k)
        
    # 4. Equation 5.2 - Calculate the overall Mean RRMS across M curves
    mean_rrms = np.mean(rrms_k_list)
    
    # Bonus: Calculate population standard deviation (Eq. 5.3)
    std_rrms = np.std(rrms_k_list)
    
    return mean_rrms, std_rrms

# 2. Initialize lists for grouping M curves
all_measured_curves = []
all_modeled_curves = []

# 3. List all your data files 
# (Option A: Hardcoded list if names are unique)
path1 = '/home/oliviag/ngspice-skywater-sims/vgsSweep_out/'
file_list_model = ['vgs_sweep_step0.txt', 'vgs_sweep_step1.txt', 'vgs_sweep_step2.txt','vgs_sweep_step3.txt','vgs_sweep_step4.txt','vgs_sweep_step5.txt']
modFile = [os.path.join(path1, filename) for filename in file_list_model]

path2 = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/nmos_FET_len_0p15_wid_1p6/'
file_list_meas = ['idvg_Vd0p01.csv','idvg_Vd0p37.csv','idvg_Vd0p74.csv','idvg_Vd1p11.csv','idvg_Vd1p48.csv', 'idvg_Vd1p85.csv']
measFile = [os.path.join(path2, filename) for filename in file_list_meas]

path3 = '/home/oliviag/ngspice-skywater-sims/vdsSweep_out/'
file_list_modelD = ['vds_sweep_step0.txt', 'vds_sweep_step1.txt', 'vds_sweep_step2.txt','vds_sweep_step3.txt','vds_sweep_step4.txt']
modFileD = [os.path.join(path3, filename) for filename in file_list_modelD]

path4 = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/nmos_FET_len_0p15_wid_1p6/'
file_list_measD = ['idvd_Vg0p37.csv','idvd_Vg0p74.csv','idvd_Vg1p11.csv','idvd_Vg1p48.csv','idvd_Vg1p85.csv']
measFileD = [os.path.join(path4, filename) for filename in file_list_measD]


for mPath in modFile:
    try:
       
        y_model = np.loadtxt(mPath, skiprows=1, usecols=(3), unpack=True)
        
        #all_measured_curves.append(y_data)2
        all_modeled_curves.append(y_model)
        #print(f"Successfully loaded: {mPath} ({len(y_data)} points)")
        
    except Exception as e:
        print(f"Error loading {mPath}: {e}")
for measPath in measFile:
    try:
        # skiprows=1 skips the header line if your file has one
        # unpack=True splits columns into separate arrays
        # usecols=(1, 2) extracts column 1 (measured) and column 2 (modeled)
        y_data = np.loadtxt(measPath, skiprows=1, delimiter=',', usecols=(2), unpack=True)
        
        all_measured_curves.append(y_data)
        #all_modeled_curves.append(y_model)
        #print(f"Successfully loaded: {file_path} ({len(y_data)} points)")
        
    except Exception as e:
        print(f"Error loading {measPath}: {e}")
for mPath in modFileD:
    try:
       
        y_modelD = np.loadtxt(mPath, skiprows=1, usecols=(3), unpack=True)
        
        #all_measured_curves.append(y_data)2
        all_modeled_curves.append(y_modelD)
        #print(f"Successfully loaded: {mPath} ({len(y_data)} points)")
        
    except Exception as e:
        print(f"Error loading {mPath}: {e}")
for measPath in measFileD:
    try:
        # skiprows=1 skips the header line if your file has one
        # unpack=True splits columns into separate arrays
        # usecols=(1, 2) extracts column 1 (measured) and column 2 (modeled)
        y_dataD = np.loadtxt(measPath, skiprows=1, delimiter=',', usecols=(2), unpack=True)
        
        all_measured_curves.append(y_dataD)
        #all_modeled_curves.append(y_model)
        #print(f"Successfully loaded: {file_path} ({len(y_data)} points)")
        
    except Exception as e:
        print(f"Error loading {measPath}: {e}")

# 5. Calculate and print final result
if all_measured_curves:
    # Unpack the tuple into two separate variables
    final_rrms, final_std = calculate_overall_rrms(all_measured_curves, all_modeled_curves)     
    
    # Now you can safely format them as floats
    print(f"\nOverall Mean RRMS: {final_rrms:.4f}")     
    print(f"Population Std Dev (σ): {final_std:.4f}")
else:
    print("No data files were loaded.")



Overall Mean RRMS: 0.0678
Population Std Dev (σ): 0.0710


In [1]:
import numpy as np
import os

def calculate_overall_rrms(measured_list, modeled_list):
    """
    Computes overall RRMS by averaging the individual RRMS errors 
    of M distinct curves: RRMS = (1/M) * sum(RRMS_k)
    """
    rrms_k_list = []
    M = len(measured_list)
    
    if M == 0:
        return 0.0, 0.0
        
    for k in range(M):
        meas = np.array(measured_list[k])
        mod = np.array(modeled_list[k])
        
        # 1. Compute the RMS Error for curve k (Numerator)
        rms_error_k = np.sqrt(np.mean((mod - meas) ** 2))
        
        # 2. Compute Mean of Data for curve k (Denominator)
        mean_data_k = np.mean(np.abs(meas))
        if mean_data_k == 0:
            mean_data_k = 1e-9
            
        # 3. Calculate RRMS_k
        rrms_k = rms_error_k / mean_data_k
        rrms_k_list.append(rrms_k)
    
    # 4. Perform the final summation over M curves and divide by M
    overall_mean_rrms = (1.0 / M) * np.sum(rrms_k_list)
    population_std = np.std(rrms_k_list)
    
    return overall_mean_rrms, population_std


# Initialize data pools
all_measured_curves = []
all_modeled_curves = []

# Define file paths and pairs
path_vgs_mod  = '/home/oliviag/ngspice-skywater-sims/vgsSweep_out/'
path_vgs_meas = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/nmos_FET_len_0p15_wid_1p6/'
vgs_pairs = [
    ('vgs_sweep_step0.txt', 'idvg_Vd0p01.csv'),
    ('vgs_sweep_step1.txt', 'idvg_Vd0p37.csv'),
    ('vgs_sweep_step2.txt', 'idvg_Vd0p74.csv'),
    ('vgs_sweep_step3.txt', 'idvg_Vd1p11.csv'),
    ('vgs_sweep_step4.txt', 'idvg_Vd1p48.csv'),
    ('vgs_sweep_step5.txt', 'idvg_Vd1p85.csv')
]

path_vds_mod  = '/home/oliviag/ngspice-skywater-sims/vdsSweep_out/'
path_vds_meas = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/nmos_FET_len_0p15_wid_1p6/'
vds_pairs = [
    ('vds_sweep_step0.txt', 'idvd_Vg0p37.csv'),
    ('vds_sweep_step1.txt', 'idvd_Vg0p74.csv'),
    ('vds_sweep_step2.txt', 'idvd_Vg1p11.csv'),
    ('vds_sweep_step3.txt', 'idvd_Vg1p48.csv'),
    ('vds_sweep_step4.txt', 'idvd_Vg1p85.csv')
]

# --- Load VGS Sweep Curves ---
for mod_f, meas_f in vgs_pairs:
    m_path = os.path.join(path_vgs_mod, mod_f)
    p_path = os.path.join(path_vgs_meas, meas_f)
    try:
        y_model = np.loadtxt(m_path, usecols=3, unpack=True)
        y_meas  = np.loadtxt(p_path, skiprows=1, delimiter=',', usecols=2, unpack=True)
        
        all_modeled_curves.append(y_model)
        all_measured_curves.append(y_meas)
    except Exception as e:
        print(f"⚠️ Skipping pair {mod_f} / {meas_f} due to error: {e}")

# --- Load VDS Sweep Curves ---
for mod_f, meas_f in vds_pairs:
    m_path = os.path.join(path_vds_mod, mod_f)
    p_path = os.path.join(path_vds_meas, meas_f)
    try:
        y_model = np.loadtxt(m_path, usecols=3, unpack=True)
        y_meas  = np.loadtxt(p_path, skiprows=1, delimiter=',', usecols=2, unpack=True)
        
        all_modeled_curves.append(y_model)
        all_measured_curves.append(y_meas)
    except Exception as e:
        print(f"⚠️ Skipping pair {mod_f} / {meas_f} due to error: {e}")

# --- Compute Metrics ---
if all_measured_curves:
    final_rrms, final_std = calculate_overall_rrms(all_measured_curves, all_modeled_curves)     
    print(f"\nOverall Mean RRMS: {final_rrms:.4f}")     
    print(f"Population Std Dev (σ): {final_std:.4f}")
else:
    print("No data files were successfully loaded.")


Overall Mean RRMS: 0.0597
Population Std Dev (σ): 0.0739


In [16]:
p_measured_curves = []
p_modeled_curves = []

# Define file paths and pairs
#path_pvgs_mod  = '/home/oliviag/ngspice-skywater-sims/vgsSweep_out/'
path_pvgs_meas = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/pmos_FET_len_8_wid_0p84/'
vgs_pairs = [
    ('p_vgs_sweep_step0.txt', 'idvg_Vd-0p01.csv'),
    ('p_vgs_sweep_step1.txt', 'idvg_Vd-0p37.csv'),
    ('p_vgs_sweep_step2.txt', 'idvg_Vd-0p74.csv'),
    ('p_vgs_sweep_step3.txt', 'idvg_Vd-1p11.csv'),
    ('p_vgs_sweep_step4.txt', 'idvg_Vd-1p48.csv'),
    ('p_vgs_sweep_step5.txt', 'idvg_Vd-1p85.csv')
]

#path_pvds_mod  = '/home/oliviag/ngspice-skywater-sims/vdsSweep_out/'
path_pvds_meas = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/pmos_FET_len_8_wid_0p84/'
vds_pairs = [
    ('p_vds_sweep_step0.txt', 'idvd_Vg-0p37.csv'),
    ('p_vds_sweep_step1.txt', 'idvd_Vg-0p74.csv'),
    ('p_vds_sweep_step2.txt', 'idvd_Vg-1p11.csv'),
    ('p_vds_sweep_step3.txt', 'idvd_Vg-1p48.csv'),
    ('p_vds_sweep_step4.txt', 'idvd_Vg-1p85.csv')
]

# --- Load VGS Sweep Curves ---
for mod_f, meas_f in vgs_pairs:
    m_path = os.path.join(path_vgs_mod, mod_f)
    p_path = os.path.join(path_pvgs_meas, meas_f)
    try:
        p_model = np.loadtxt(m_path, usecols=3, unpack=True)
        p_meas  = np.loadtxt(p_path, skiprows=1, delimiter=',', usecols=2, unpack=True)
        
        p_modeled_curves.append(p_model)
        p_measured_curves.append(p_meas)
    except Exception as e:
        print(f"⚠️ Skipping pair {mod_f} / {meas_f} due to error: {e}")

# --- Load VDS Sweep Curves ---
for mod_f, meas_f in vgs_pairs:
    m_path = os.path.join(path_vgs_mod, mod_f)
    p_path = os.path.join(path_pvgs_meas, meas_f)
    try:
        p_model = np.loadtxt(m_path, usecols=3, unpack=True)
        p_meas = np.loadtxt(p_path, skiprows=1, delimiter=',', usecols=2, unpack=True)
        
        # FIX: Force both to absolute values to ignore SPICE terminal sign conventions
        p_modeled_curves.append(np.abs(p_model))
        p_measured_curves.append(np.abs(p_meas))
    except Exception as e:
        print(f"⚠️ Skipping pair {mod_f} / {meas_f} due to error: {e}")

# --- Compute Metrics ---
if all_measured_curves:
    final_rrms, final_std = calculate_overall_rrms(p_measured_curves, p_modeled_curves)     
    print(f"\nOverall Mean RRMS: {final_rrms:.4f}")     
    print(f"Population Std Dev (σ): {final_std:.4f}")
else:
    print("No data files were successfully loaded.")


Overall Mean RRMS: 0.7709
Population Std Dev (σ): 1.2703


In [8]:
#all Pfet RRMS
import os
import numpy as np

# ==========================================
# 1. DIRECTORY & FILE DEFINITIONS
# ==========================================
pMeas_Dir = '/home/oliviag/Skywater-130nm-77K-Cryogenic-Models/cryo_data/'
pModel_Dir = '/home/oliviag/ngspice-skywater-sims/pfet_mod/'

# Main data lists containing ALL matching datasets structurally together
p_device_names = []
p_measured_curves_all = []
p_modeled_curves_all = []

p_RRMS_indv = []

# Mappings for Vd Sweep Directories
pairs_vd = [
    ('pmos_FET_len_8_wid_0p84', 'l_8p0_w_0p84/vd_sweep/'),
    ('pmos_FET_len_0.35_wid_1p6', 'l_0p35_w_1p6/vd_sweep/'),
    ('pmos_FET_len_0p5_wid_0p42', 'l_0p50_w_0p42/vd_sweep/'),
    ('pmos_FET_len_0.35_wid_0.55', 'l_0p35_w_0p55/vd_sweep/'),
    ('pmos_FET_len_0p5_wid_0p64', 'l_0p50_w_0p64/vd_sweep/'),
    ('pmos_FET_len_0.35_wid_5', 'l_0p35_w_5p0/vd_sweep/'),
    ('pmos_FET_len_2_wid_5', 'l_2p0_w_5p0/vd_sweep/'),
    ('pmos_FET_len_4_wid_7', 'l_4p0_w_7p0/vd_sweep/'),
    ('pmos_FET_len_8_wid_1p6', 'l_8p0_w_1p6/vd_sweep/'),
    ('pmos_FET_len_8_wid_5', 'l_8p0_w_5p0/vd_sweep/')
]

# Mappings for Vg Sweep Directories
pairs_vg = [
    ('pmos_FET_len_8_wid_0p84', 'l_8p0_w_0p84/vg_sweep/'),
    ('pmos_FET_len_0.35_wid_1p6', 'l_0p35_w_1p6/vg_sweep/'),
    ('pmos_FET_len_0p5_wid_0p42', 'l_0p50_w_0p42/vg_sweep/'),
    ('pmos_FET_len_0.35_wid_0.55', 'l_0p35_w_0p55/vg_sweep/'),
    ('pmos_FET_len_0p5_wid_0p64', 'l_0p50_w_0p64/vg_sweep/'),
    ('pmos_FET_len_0.35_wid_5', 'l_0p35_w_5p0/vg_sweep/'),
    ('pmos_FET_len_2_wid_5', 'l_2p0_w_5p0/vg_sweep/'),
    ('pmos_FET_len_4_wid_7', 'l_4p0_w_7p0/vg_sweep/'),
    ('pmos_FET_len_8_wid_1p6', 'l_8p0_w_1p6/vg_sweep/'),
    ('pmos_FET_len_8_wid_5', 'l_8p0_w_5p0/vg_sweep/')
]

# (Ngspice Output Name, Experimental CSV Name)
vds_pairs = [
    ('p_vds_sweep_step0.txt', 'idvd_Vg-0p37.csv'),
    ('p_vds_sweep_step1.txt', 'idvd_Vg-0p74.csv'),
    ('p_vds_sweep_step2.txt', 'idvd_Vg-1p11.csv'),
    ('p_vds_sweep_step3.txt', 'idvd_Vg-1p48.csv'),
    ('p_vds_sweep_step4.txt', 'idvd_Vg-1p85.csv')
]

vgs_pairs = [
    ('p_vgs_sweep_step0.txt', 'idvg_Vd-0p01.csv'),
    ('p_vgs_sweep_step1.txt', 'idvg_Vd-0p37.csv'),
    ('p_vgs_sweep_step2.txt', 'idvg_Vd-0p74.csv'),
    ('p_vgs_sweep_step3.txt', 'idvg_Vd-1p11.csv'),
    ('p_vgs_sweep_step4.txt', 'idvg_Vd-1p48.csv'),
    ('p_vgs_sweep_step5.txt', 'idvg_Vd-1p85.csv')
]


# ==========================================
# 2. DATA PROCESSING AND LIST ASSEMBLY
# ==========================================
print("🔄 Loading data files into master arrays...")

for i in range(len(pairs_vd)):
    measDir, modDirVd = pairs_vd[i]
    _, modDirVg = pairs_vg[i]
    
    # Target structures for this exact iteration's transistor
    this_device_meas = []
    this_device_mod = []
    
    pathMeas = os.path.join(pMeas_Dir, measDir)
    pathModVd = os.path.join(pModel_Dir, modDirVd)
    pathModVg = os.path.join(pModel_Dir, modDirVg)
    
    # Extract Vg curves
    for mod_f, meas_f in vgs_pairs:
        try:
            p_model = np.loadtxt(os.path.join(pathModVg, mod_f), usecols=3, unpack=True)
            p_meas = np.loadtxt(os.path.join(pathMeas, meas_f), skiprows=1, delimiter=',', usecols=2, unpack=True)
            this_device_mod.append(np.abs(p_model))
            this_device_meas.append(np.abs(p_meas))
        except Exception as e:
            pass  # Skips missing points quietly to keep loop clean

    # Extract Vd curves
    for mod_f, meas_f in vds_pairs:
        try:
            p_model = np.loadtxt(os.path.join(pathModVd, mod_f), usecols=3, unpack=True)
            p_meas = np.loadtxt(os.path.join(pathMeas, meas_f), skiprows=1, delimiter=',', usecols=2, unpack=True)
            this_device_mod.append(np.abs(p_model))
            this_device_meas.append(np.abs(p_meas))
        except Exception as e:
            pass

    # Save finalized data groupings into the structured master arrays
    if this_device_meas:
        p_device_names.append(measDir)
        p_measured_curves_all.append(this_device_meas)
        p_modeled_curves_all.append(this_device_mod)


# ==========================================
# 3. MATHEMATICAL RRMS ENGINE
# ==========================================
def calculate_rrms(meas_curves, mod_curves):
    """Computes the root-mean-square error normalized by total signal power."""
    total_sq_error = 0
    total_sq_meas = 0
    
    for meas, mod in zip(meas_curves, mod_curves):
        # Clip edge mismatches if data length doesn't perfectly align
        min_len = min(len(meas), len(mod))
        meas = meas[:min_len]
        mod = mod[:min_len]
        
        total_sq_error += np.sum((meas - mod) ** 2)
        total_sq_meas += np.sum(meas ** 2)
        
    return np.sqrt(total_sq_error / total_sq_meas) if total_sq_meas > 0 else 0.0


# ==========================================
# 4. EXECUTE ANALYSIS PER DEVICE
# ==========================================
for idx in range(len(p_device_names)):
    device = p_device_names[idx]
    meas_list = p_measured_curves_all[idx]
    mod_list = p_modeled_curves_all[idx]
    
    device_rrms = calculate_rrms(meas_list, mod_list)
    p_RRMS_indv.append({
        'geometry': device, 
        'rrms': device_rrms
    })


# ==========================================
# 5. METRIC REPORTS AND POPULATION AVERAGES
# ==========================================
print("\n" + "="*60)
print(f"{'SKYWATER PMOS GEOMETRY':<35} | {'COMBINED RRMS':<15}")
print("="*60)
for entry in p_RRMS_indv:
    print(f"{entry['geometry']:<35} | {entry['rrms']:.4f}")

if p_RRMS_indv:
    # Isolate raw calculations to find population metrics
    all_rrms_values = [entry['rrms'] for entry in p_RRMS_indv]
    
    mean_rrms = np.mean(all_rrms_values)
    std_rrms = np.std(all_rrms_values)
    
    print("="*60)
    print(f"{'POPULATION MEAN RRMS:':<35} | {mean_rrms:.4f}")
    print(f"{'POPULATION STANDARD DEV (σ):':<35} | {std_rrms:.4f}")
    print("="*60)
else:
    print("\n❌ Error: No datasets were successfully parsed and computed.")




🔄 Loading data files into master arrays...

SKYWATER PMOS GEOMETRY              | COMBINED RRMS  
pmos_FET_len_8_wid_0p84             | 0.3876
pmos_FET_len_0.35_wid_1p6           | 1.1598
pmos_FET_len_0p5_wid_0p42           | 0.5054
pmos_FET_len_0p5_wid_0p64           | 0.5521
pmos_FET_len_2_wid_5                | 0.4306
pmos_FET_len_4_wid_7                | 0.3977
pmos_FET_len_8_wid_1p6              | 0.3780
pmos_FET_len_8_wid_5                | 0.3826
POPULATION MEAN RRMS:               | 0.5242
POPULATION STANDARD DEV (σ):        | 0.2475
